# 画面A・Bの操作時間を比べる

動態デザイン研究室「統計解析入門」の練習ノートです。12人が両方の画面を使った架空データを使います。

[教材へ](https://design-for-changes.github.io/learning/#/statistics/python)

自分の作業用コピーを保存して進めてください。

## 1. CSVを用意する

[ui_comparison.csvをダウンロード](https://design-for-changes.github.io/learning/data/ui_comparison.csv)し、次のセルでアップロードします。1行が1人、A・Bの列が操作時間（秒）です。

In [ ]:
from google.colab import files
files.upload()

## 2. 自分で試す

教材のプロンプトでAIにコードの書き方を相談し、下のセルから実行してください。表の読み込み、図、検定を一度に行わず、途中の結果を確かめます。セルは必要に応じて追加できます。

In [ ]:
# AIと相談して作ったコードを、ここから試してください。


## 3. 考えてみる

- Bの方が速かったのは全員？
- 平均差と95%信頼区間は、何の差？
- このデータだけで「誰でもBの方が使いやすい」と言える？

ここに自分の言葉で説明を書いてみてください。

## 4. 確認用コード

ここからは、教材と計算結果を照らし合わせるためのコードです。練習では参加者間の独立性と、差の母集団分布の正規性を仮定します。実施順序の影響を分離した解析ではありません。

In [ ]:
import sys
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

df = pd.read_csv("ui_comparison.csv")
print(df.head())
print("行数・列数:", df.shape)
print("欠損数:\n", df.isna().sum())
print("重複した参加者ID:", df["participant_id"].duplicated().sum())

a = df["time_A_sec"]
b = df["time_B_sec"]
assert not df["participant_id"].duplicated().any(), "参加者IDの重複を確認してください"
assert a.notna().all() and b.notna().all(), "欠損の扱いを検討してください"
assert (a >= 0).all() and (b >= 0).all(), "時間の入力を確認してください"
diff = a - b  # 正の値なら、Bの方が短い
print("平均時間 A / B:", a.mean(), b.mean())
print("中央値 A / B:", a.median(), b.median())
print("標本標準偏差 A / B:", a.std(ddof=1), b.std(ddof=1))
print("Bが速い / 遅い / 同じ人数:", (diff > 0).sum(), (diff < 0).sum(), (diff == 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for va, vb in zip(a, b):
    axes[0].plot([0, 1], [va, vb], "o-", alpha=0.6)
axes[0].set(xticks=[0, 1], xticklabels=["A", "B"], ylabel="Time (seconds)", title="Each participant")
axes[1].scatter(range(1, len(diff) + 1), diff)
axes[1].axhline(0, color="gray", linestyle="--")
axes[1].set(xlabel="Participant", ylabel="A - B (seconds)", title="Within-person difference")
plt.tight_layout()
plt.show()



In [ ]:
# 練習では、参加者間の独立性と、差の母集団分布の正規性を仮定する
result = stats.ttest_rel(a, b, alternative="two-sided")
ci = result.confidence_interval(confidence_level=0.95)
print(f"平均差 A-B: {diff.mean():.2f} 秒")
print(f"95%信頼区間: {ci.low:.2f} 〜 {ci.high:.2f} 秒")
print(f"t({int(result.df)}) = {result.statistic:.3f}, p = {result.pvalue:.4f}")
print("Python / pandas / SciPy / Matplotlib:", sys.version.split()[0], pd.__version__, scipy.__version__, matplotlib.__version__)


## 結果の目安

平均差A−Bは5.00秒、95%信頼区間は0.42〜9.58秒。t(11)=2.402、p=0.0351。Bの方が速い8人、遅い3人、同じ1人。

値が違ったら、差の向き・対応の保持・欠測や除外・片側と両側を確認してください。有意差だけで使いやすさ全体を結論づけないでください。